# EDA — NYC Yellow Taxi (couche Silver)

Analyse exploratoire des données de courses de taxi new-yorkaises, à partir de la couche **Silver** (données nettoyées) produite par `spark/silver_job.py`.

**Prérequis :**
- Le cluster Spark (`spark-master` + `spark-worker`) et HDFS (`namenode` + `datanode`) doivent être démarrés.
- Ce notebook doit tourner dans un environnement ayant accès au réseau Docker `taxi-net` (par ex. un conteneur Jupyter connecté au même réseau, ou en exécutant ce notebook depuis un conteneur Spark).
- Si tu exécutes ce notebook en dehors de Docker, adapte `SILVER_PATH` vers un chemin local (copie préalable des fichiers Parquet).

In [ ]:
import pyspark
from pyspark.sql import SparkSession, functions as F
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

SILVER_PATH = "hdfs://namenode:9000/data/silver/taxi"
SAMPLE_FRACTION = 0.05  # échantillon pour les visualisations (évite de tout charger en mémoire)

In [ ]:
spark = (
    SparkSession.builder
    .appName("EDA-TaxiSilver")
    .master("spark://spark-master:7077")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

df = spark.read.parquet(SILVER_PATH)
print(f"Nombre de lignes : {df.count():,}")
print(f"Nombre de colonnes : {len(df.columns)}")
df.printSchema()

## 1. Statistiques descriptives

In [ ]:
num_cols = [
    "trip_distance", "fare_amount", "tip_amount", "tolls_amount",
    "total_amount", "passenger_count", "trip_duration_minutes", "price_per_mile",
]
df.select(num_cols).describe().toPandas()

## 2. Valeurs manquantes

In [ ]:
missing = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns
]).toPandas().T
missing.columns = ["missing_count"]
missing[missing["missing_count"] > 0].sort_values("missing_count", ascending=False)

## 3. Échantillonnage pour les visualisations\n\nLa couche Silver peut représenter plusieurs millions de lignes : on travaille sur un échantillon pour les graphiques (les KPIs exacts, eux, sont calculés sur l'ensemble des données dans la couche Gold).

In [ ]:
pdf = df.sample(fraction=SAMPLE_FRACTION, seed=42).toPandas()
print(f"Taille de l'échantillon : {len(pdf):,} lignes")
pdf.head()

## 4. Distribution des prix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(pdf["total_amount"], bins=60, kde=True, ax=axes[0])
axes[0].set_title("Distribution du prix total (total_amount)")
axes[0].set_xlabel("Prix ($)")

sns.boxplot(x=pdf["total_amount"], ax=axes[1])
axes[1].set_title("Boxplot du prix total")
axes[1].set_xlabel("Prix ($)")

plt.tight_layout()
plt.show()

## 5. Distribution des distances

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(pdf["trip_distance"], bins=60, kde=True, ax=axes[0])
axes[0].set_title("Distribution de la distance (miles)")
axes[0].set_xlabel("Distance (miles)")

sns.boxplot(x=pdf["trip_distance"], ax=axes[1])
axes[1].set_title("Boxplot de la distance")
axes[1].set_xlabel("Distance (miles)")

plt.tight_layout()
plt.show()

## 6. Répartition des courses selon l'heure et le jour

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(x="pickup_hour", data=pdf, color="steelblue", ax=axes[0])
axes[0].set_title("Nombre de courses par heure de la journée")
axes[0].set_xlabel("Heure")

day_labels = {1: "Dim", 2: "Lun", 3: "Mar", 4: "Mer", 5: "Jeu", 6: "Ven", 7: "Sam"}
pdf["day_label"] = pdf["pickup_dayofweek"].map(day_labels)
order = ["Lun", "Mar", "Mer", "Jeu", "Ven", "Sam", "Dim"]
sns.countplot(x="day_label", data=pdf, order=order, color="darkorange", ax=axes[1])
axes[1].set_title("Nombre de courses par jour de la semaine")
axes[1].set_xlabel("Jour")

plt.tight_layout()
plt.show()

## 7. Prix moyen selon l'heure

In [ ]:
hourly_avg = (
    df.groupBy("pickup_hour")
    .agg(F.avg("total_amount").alias("avg_price"), F.count("*").alias("nb_trips"))
    .orderBy("pickup_hour")
    .toPandas()
)

plt.figure(figsize=(10, 5))
sns.lineplot(x="pickup_hour", y="avg_price", data=hourly_avg, marker="o")
plt.title("Prix moyen par heure de la journée")
plt.xlabel("Heure")
plt.ylabel("Prix moyen ($)")
plt.show()

## 8. Top zones de départ et d'arrivée

In [ ]:
top_pickup = (
    df.groupBy("PULocationID").count()
    .orderBy(F.desc("count"))
    .limit(15)
    .toPandas()
)

plt.figure(figsize=(10, 6))
sns.barplot(x="count", y="PULocationID", data=top_pickup, orient="h", color="mediumseagreen")
plt.title("Top 15 zones de départ (par ID de zone TLC)")
plt.xlabel("Nombre de courses")
plt.ylabel("PULocationID")
plt.show()

## 9. Corrélations

In [ ]:
corr_cols = ["trip_distance", "trip_duration_minutes", "passenger_count", "tip_amount", "total_amount"]
corr_matrix = pdf[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f")
plt.title("Heatmap des corrélations")
plt.show()

## 10. Synthèse

À compléter après lecture des graphiques ci-dessus :
- Quels facteurs semblent le plus corrélés au prix ?
- Y a-t-il des patterns horaires ou hebdomadaires clairs ?
- Quelles zones concentrent le plus de départs/arrivées ?

Ces observations orienteront le choix des variables explicatives pour le modèle de Machine Learning (`spark/ml_training.py`).

In [ ]:
spark.stop()